In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np
import random
import math
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [2]:
print("Downloading Europarl dataset")
!wget -q -O europarl.zip https://object.pouta.csc.fi/OPUS-Europarl/v7/moses/en-fr.txt.zip

print("Extracting")
!unzip -q -o europarl.zip

print("Loading English sentences")
eng_sentences = []
with open('Europarl.en-fr.en', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 150000:
            break
        eng = line.strip().lower()
        if len(eng.split()) <= 50:
            eng_sentences.append(eng)

print(f"Loaded {len(eng_sentences)} English sentences")

print("Loading French sentences")
fra_sentences = []
with open('Europarl.en-fr.fr', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 150000:
            break
        fra = line.strip().lower()
        if len(fra.split()) <= 50:
            fra_sentences.append(fra)

print(f"Loaded {len(fra_sentences)} French sentences")

min_len = min(len(eng_sentences), len(fra_sentences))
eng_sentences = eng_sentences[:min_len]
fra_sentences = fra_sentences[:min_len]

print(f"\nFinal paired sentences: {len(eng_sentences)}")
print(f"Example English: {eng_sentences[0]}")
print(f"Example French: {fra_sentences[0]}")

Extracting
Loading English sentences
Loaded 139466 English sentences
Loading French sentences
Loaded 137689 French sentences

Final paired sentences: 137689
Example English: resumption of the session
Example French: reprise de la session


In [3]:
from collections import Counter

def build_vocab(sentences, min_freq=2, max_size=20000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.split())

    filtered = {word: count for word, count in counter.items() if count >= min_freq}
    most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)[:max_size]

    word2idx = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
    idx2word = {0: '<PAD>', 1: '<UNK>', 2: '<SOS>', 3: '<EOS>'}

    for idx, (word, _) in enumerate(most_common, start=4):
        word2idx[word] = idx
        idx2word[idx] = word

    return word2idx, idx2word

print("Building English vocabulary")
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
print(f"English vocab size: {len(eng_word2idx)}")

print("Building French vocabulary")
fra_word2idx, fra_idx2word = build_vocab(fra_sentences)
print(f"French vocab size: {len(fra_word2idx)}")

Building English vocabulary
English vocab size: 20004
Building French vocabulary
French vocab size: 20004


In [4]:
def encode_sentence(sentence, word2idx, max_len=50):
    tokens = sentence.split()
    indices = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

    if len(indices) > max_len - 1:
        indices = indices[:max_len - 1]

    indices.append(word2idx['<EOS>'])
    indices = indices + [word2idx['<PAD>']] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, eng, fra, eng_w2i, fra_w2i, max_len=50):
        self.eng = eng
        self.fra = fra
        self.eng_w2i = eng_w2i
        self.fra_w2i = fra_w2i
        self.max_len = max_len

    def __len__(self):
        return len(self.eng)

    def __getitem__(self, idx):
        eng = encode_sentence(self.eng[idx], self.eng_w2i, self.max_len)
        fra = encode_sentence(self.fra[idx], self.fra_w2i, self.max_len)
        return eng, fra

split = int(0.95 * len(eng_sentences))
train_eng, train_fra = eng_sentences[:split], fra_sentences[:split]
val_eng, val_fra = eng_sentences[split:], fra_sentences[split:]

print(f"Train pairs: {len(train_eng)}")
print(f"Val pairs: {len(val_eng)}")

max_len = 50
batch_size = 64

train_dataset = TranslationDataset(train_eng, train_fra, eng_word2idx, fra_word2idx, max_len)
val_dataset = TranslationDataset(val_eng, val_fra, eng_word2idx, fra_word2idx, max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train pairs: 130804
Val pairs: 6885
Train batches: 2044
Val batches: 108


In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = hidden.view(self.num_layers, 2, -1, self.hidden_dim)
        hidden = hidden.permute(1, 0, 2, 3).contiguous()
        hidden = hidden.view(2, -1, self.hidden_dim * 2)
        cell = cell.view(self.num_layers, 2, -1, self.hidden_dim)
        cell = cell.permute(1, 0, 2, 3).contiguous()
        cell = cell.view(2, -1, self.hidden_dim * 2)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim * 2, num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden_dim * 4, 1)
        self.fc_out = nn.Linear(hidden_dim * 4, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim

    def forward(self, x, hidden, cell, encoder_outputs):
        if x.dim() == 1:
            x = x.unsqueeze(1)

        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        seq_len = encoder_outputs.shape[1]
        hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        concat = torch.cat((hidden_repeated, encoder_outputs), dim=2)
        attention_weights = torch.softmax(self.attention(concat), dim=1)
        context = torch.sum(attention_weights * encoder_outputs, dim=1).unsqueeze(1)

        concat_final = torch.cat((output, context), dim=2)
        prediction = self.fc_out(concat_final)
        return prediction.squeeze(1), hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = trg.shape[1]
        batch_size = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

embedding_dim = 128
hidden_dim = 256
dropout = 0.3
num_layers = 2

encoder = Encoder(len(eng_word2idx), embedding_dim, hidden_dim, num_layers, dropout)
decoder = Decoder(len(fra_word2idx), embedding_dim, hidden_dim, num_layers, dropout)
model = Seq2Seq(encoder, decoder, device).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"Embedding dim: {embedding_dim}, Hidden dim: {hidden_dim}, Dropout: {dropout}")

Model has 31,409,701 trainable parameters
Embedding dim: 128, Hidden dim: 256, Dropout: 0.3


In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

train_losses = []
val_losses = []

def train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0
    for batch_idx, (src, tgt) in enumerate(loader):
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio)

        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 200 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)

            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            total_loss += loss.item()

    return total_loss / len(loader)

print("Training setup complete")
print(f"Optimizer: Adam (lr=0.001, weight_decay=1e-5)")
print(f"Loss: CrossEntropyLoss (ignore PAD)")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")

Training setup complete
Optimizer: Adam (lr=0.001, weight_decay=1e-5)
Loss: CrossEntropyLoss (ignore PAD)
Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)


In [ ]:
epochs = 35
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 7

print(f"Starting training for {epochs} epochs on {len(train_eng)} sentences")

for epoch in range(epochs):
    teacher_ratio = max(0.3, 0.9 - (epoch * 0.02))

    train_loss = train_epoch(model, train_loader, optimizer, criterion, teacher_ratio)
    val_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"LR: {current_lr:.6f}")
    print(f"Teacher Forcing: {teacher_ratio:.2f}")

    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f'checkpoint_epoch_{epoch+1}.pt')
        print(f" Checkpoint saved: checkpoint_epoch_{epoch+1}.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_final.pt')
        print(f" Saved best model (val_loss: {val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f" No improvement. Patience: {patience_counter}/{early_stop_patience}")

    if patience_counter >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved as: best_model_final.pt")

In [ ]:
model.load_state_dict(torch.load('best_model_final.pt', map_location=device))
model.eval()
print("Loaded best trained model")

def beam_search_decode(model, src_sentence, eng_word2idx, fra_idx2word, fra_word2idx, beam_width=5, max_len=50):
    tokens = src_sentence.lower().split()
    indices = [eng_word2idx.get(token, eng_word2idx['<UNK>']) for token in tokens]
    if len(indices) > max_len - 1:
        indices = indices[:max_len - 1]
    indices.append(eng_word2idx['<EOS>'])
    indices = indices + [eng_word2idx['<PAD>']] * (max_len - len(indices))
    src_tensor = torch.tensor(indices).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

        start_idx = fra_word2idx['<SOS>']
        eos_idx = fra_word2idx['<EOS>']

        beams = [([start_idx], 0.0, hidden, cell)]

        for _ in range(max_len):
            all_candidates = []
            for seq, score, hidden, cell in beams:
                if seq[-1] == eos_idx:
                    all_candidates.append((seq, score, hidden, cell))
                    continue

                tgt_tensor = torch.tensor([seq[-1]]).to(device)
                output, new_hidden, new_cell = model.decoder(tgt_tensor, hidden, cell, encoder_outputs)
                log_probs = torch.log_softmax(output, dim=-1)

                top_log_probs, top_indices = log_probs.topk(beam_width, dim=-1)
                for i in range(beam_width):
                    next_token = top_indices[0][i].item()
                    next_score = score + top_log_probs[0][i].item()
                    all_candidates.append((seq + [next_token], next_score, new_hidden, new_cell))

            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beams = all_candidates[:beam_width]

            if all(seq[-1] == eos_idx for seq, _, _, _ in beams):
                break

        best_seq = beams[0][0]
        translation = []
        for idx in best_seq[1:]:
            if idx == eos_idx:
                break
            translation.append(fra_idx2word.get(idx, '<UNK>'))

        return ' '.join(translation)

test_sentences = [
    "resumption of the session",
    "thank you for your attention",
    "i agree with the proposal",
    "climate change is a serious problem",
    "the european parliament meets today"
]

print("Beam Search Translations (beam_width=5):")
for sent in test_sentences:
    trans = beam_search_decode(model, sent, eng_word2idx, fra_idx2word, fra_word2idx, beam_width=5)
    print(f"EN: {sent}")
    print(f"FR: {trans}")
    print()

In [ ]:
!pip install sacrebleu -q
import sacrebleu

print("Computing BLEU score")

def compute_bleu(model, val_loader, num_samples=500):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch_idx, (src, tgt) in enumerate(val_loader):
            if len(references) >= num_samples:
                break

            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.shape[0]):
                if len(references) >= num_samples:
                    break

                src_sent = src[i]
                encoder_outputs, hidden, cell = model.encoder(src_sent.unsqueeze(0))

                trg_idx = [fra_word2idx['<SOS>']]
                for _ in range(50):
                    tgt_tensor = torch.tensor([trg_idx[-1]]).to(device)
                    output, hidden, cell = model.decoder(tgt_tensor, hidden, cell, encoder_outputs)
                    pred = output.argmax(1).item()
                    trg_idx.append(pred)
                    if pred == fra_word2idx['<EOS>']:
                        break

                ref = []
                for idx in tgt[i].tolist():
                    if idx == fra_word2idx['<EOS>']:
                        break
                    if idx not in [fra_word2idx['<SOS>'], fra_word2idx['<PAD>']]:
                        ref.append(fra_idx2word.get(idx, '<UNK>'))

                hyp = []
                for idx in trg_idx[1:-1]:
                    if idx == fra_word2idx['<PAD>']:
                        break
                    hyp.append(fra_idx2word.get(idx, '<UNK>'))

                if len(hyp) > 0 and len(ref) > 0:
                    references.append(' '.join(ref))
                    hypotheses.append(' '.join(hyp))

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score

bleu_score = compute_bleu(model, val_loader, num_samples=500)
print(f"\nBLEU-4 Score: {bleu_score:.2f}")